# 🇮🇳 IndicF5 — Hindi Voice Cloning (Kaggle)

**High-quality Hindi TTS with Voice Cloning** by AI4Bharat / IIT Madras

Supports 11 Indian languages with near-human quality!

**Before running:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. **Verify phone** in Settings for GPU access
4. Run cells with **Shift+Enter**

In [ ]:
# Cell 1: Install IndicF5 and dependencies
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q flask flask-cors pyngrok soundfile numpy

In [ ]:
# Cell 2: Load IndicF5 model
import torch
from transformers import AutoModel
import numpy as np
import soundfile as sf

print("🔄 Loading IndicF5 model...")
print(f"   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

model = AutoModel.from_pretrained("ai4bharat/IndicF5", trust_remote_code=True)

print("✅ IndicF5 loaded!")
print("   Languages: Hindi, Bengali, Tamil, Telugu, Kannada, Malayalam, Marathi, Gujarati, Punjabi, Odia, Assamese")

In [ ]:
# Cell 3: Set ngrok token
# PASTE YOUR NGROK TOKEN HERE!
NGROK_TOKEN = ""  # <-- Get from https://dashboard.ngrok.com/get-started/your-authtoken

from pyngrok import ngrok
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ ngrok token set!")
else:
    print("⚠️ Paste your ngrok token above!")

In [ ]:
# Cell 4: Start Hindi TTS Server
from flask import Flask, request, send_file, jsonify, Response
from flask_cors import CORS
import io, base64, tempfile, os, json, re

app = Flask(__name__)
CORS(app)

def split_hindi_text(text, max_chars=120):
    """Split Hindi text on sentence/clause boundaries."""
    parts = re.split(r'(?<=[।.!?,;:\n])\s*', text)
    chunks = []
    current = ""
    for p in parts:
        p = p.strip()
        if not p:
            continue
        if len(current) + len(p) + 1 <= max_chars:
            current += (" " if current else "") + p
        else:
            if current: chunks.append(current.strip())
            if len(p) > max_chars:
                words = p.split()
                sub = ""
                for w in words:
                    if len(sub) + len(w) + 1 <= max_chars:
                        sub += (" " if sub else "") + w
                    else:
                        if sub: chunks.append(sub.strip())
                        sub = w
                current = sub
            else:
                current = p
    if current: chunks.append(current.strip())
    return chunks if chunks else [text]

@app.route('/health', methods=['GET'])
def health():
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    return jsonify({"status": "ok", "model": "IndicF5", "gpu": gpu})

@app.route('/api/tts', methods=['POST'])
def generate_tts():
    data = request.json
    text = data.get('text', 'नमस्ते')
    ref_audio_b64 = data.get('ref_audio')
    ref_text = data.get('ref_text', '')  # transcript of reference audio
    stream = data.get('stream', False)

    def generate_with_progress():
        try:
            print(f"🎙️ Hindi TTS: {len(text)} chars")

            chunks = split_hindi_text(text, max_chars=120)
            total = len(chunks)
            print(f"   {total} chunks")

            yield f"data: {json.dumps({'type': 'progress', 'percent': 0, 'status': 'Preparing...'})}\n\n"

            # Save reference audio
            ref_path = None
            if ref_audio_b64:
                with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
                    f.write(base64.b64decode(ref_audio_b64))
                    ref_path = f.name
            else:
                # Use a default prompt from the model
                ref_path = '/tmp/default_prompt.wav'
                if not os.path.exists(ref_path):
                    # Generate a small default using the model's bundled prompts
                    import urllib.request
                    urllib.request.urlretrieve(
                        'https://huggingface.co/ai4bharat/IndicF5/resolve/main/prompts/HIN_F_HAPPY_00001.wav',
                        ref_path
                    )
                if not ref_text:
                    ref_text = 'हमारे देश में हर त्यौहार बड़ी धूमधाम और खुशी के साथ मनाया जाता है।'

            yield f"data: {json.dumps({'type': 'progress', 'percent': 5, 'status': 'Voice loaded'})}\n\n"

            all_audio = []
            sr = 24000
            silence = np.zeros(int(sr * 0.25), dtype=np.float32)  # 250ms gap

            for i, chunk in enumerate(chunks):
                pct = int(((i) / total) * 90) + 5
                yield f"data: {json.dumps({'type': 'progress', 'percent': pct, 'status': f'Chunk {i+1}/{total} ({len(chunk)} chars)...'})}\n\n"

                print(f"   Generating chunk {i+1}/{total}: {chunk[:60]}...")

                audio = model(
                    chunk,
                    ref_audio_path=ref_path,
                    ref_text=ref_text if ref_text else chunk
                )

                # Convert to float32 numpy array
                if isinstance(audio, torch.Tensor):
                    audio = audio.cpu().numpy()
                audio = np.array(audio, dtype=np.float32)
                if audio.dtype == np.int16:
                    audio = audio.astype(np.float32) / 32768.0
                audio = audio.squeeze()

                all_audio.append(audio)
                if i < total - 1:
                    all_audio.append(silence)

                pct = int(((i+1) / total) * 90) + 5
                yield f"data: {json.dumps({'type': 'progress', 'percent': pct, 'status': f'Chunk {i+1}/{total} done'})}\n\n"

            if ref_audio_b64 and ref_path and os.path.exists(ref_path):
                os.unlink(ref_path)

            final = np.concatenate(all_audio)

            # Normalize volume
            peak = np.max(np.abs(final))
            if peak > 0:
                final = final * (0.95 / peak)

            buffer = io.BytesIO()
            sf.write(buffer, final, sr, format='WAV')
            buffer.seek(0)
            audio_b64 = base64.b64encode(buffer.read()).decode('utf-8')

            duration = len(final) / sr
            print(f"✅ Generated {duration:.1f}s")

            yield f"data: {json.dumps({'type': 'complete', 'audio': audio_b64, 'duration': round(duration, 1)})}\n\n"

        except Exception as e:
            import traceback; traceback.print_exc()
            yield f"data: {json.dumps({'type': 'error', 'message': str(e)})}\n\n"

    if stream:
        return Response(generate_with_progress(), mimetype='text/event-stream', headers={'Cache-Control': 'no-cache'})
    else:
        try:
            ref_path = None
            if ref_audio_b64:
                with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
                    f.write(base64.b64decode(ref_audio_b64))
                    ref_path = f.name
            else:
                ref_path = '/tmp/default_prompt.wav'
                if not ref_text:
                    ref_text = 'हमारे देश में हर त्यौहार बड़ी धूमधाम और खुशी के साथ मनाया जाता है।'

            audio = model(text, ref_audio_path=ref_path, ref_text=ref_text if ref_text else text)
            if isinstance(audio, torch.Tensor):
                audio = audio.cpu().numpy()
            audio = np.array(audio, dtype=np.float32)
            if audio.dtype == np.int16:
                audio = audio.astype(np.float32) / 32768.0

            out_path = '/tmp/indicf5_output.wav'
            sf.write(out_path, audio.squeeze(), 24000, format='WAV')
            return send_file(out_path, mimetype='audio/wav')
        except Exception as e:
            return jsonify({"error": str(e)}), 500

public_url = ngrok.connect(5000)
print("\n" + "="*50)
print(f"🇮🇳 IndicF5 Hindi URL: {public_url}")
print("="*50 + "\n")

app.run(port=5000)